# Agentic AI: Reasoning, Tool Use, and Evaluation

This assignment compares four prompting strategies and a restricted calculator agent using frozen Qwen2.5-0.5B-Instruct on GSM8K. It evaluates accuracy and inference cost, analyzes reasoning failures, and examines how undocumented symptoms are handled in Ferber et al.’s medical-agent simulation.

# Preparation

In [1]:
%pip -q install -U transformers accelerate datasets "pyarrow==23.0.1"

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
train14 = load_dataset("openai/gsm8k", "main", split="train[:14]")

development = train14.select(range(0, 10))
demonstrations = train14.select(range(10, 14))

print("Development examples:", len(development))
print("Four-shot demonstrations:", len(demonstrations))

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Development examples: 10
Four-shot demonstrations: 4


In [4]:
test_questions = (
    load_dataset("openai/gsm8k", "main", split="test[:30]")
    .select_columns(["question"])
)

print("Test questions:", len(test_questions))
print("Available columns:", test_questions.column_names)

Test questions: 30
Available columns: ['question']


In [5]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)

print("Model ready")
print("Device:", model.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model ready
Device: cuda:0


# Question 1

In [21]:
import re
import torch

def extract_answer(text):
    number = r"(-?\d[\d,]*(?:\.\d+)?)"
    currency = r"(?:\\?\$[ \t]*)?"

    # Öncelik: ####, ardından boxed, ardından Final answer.
    patterns = [
        rf"####[ \t]*{currency}{number}",
        rf"\\boxed\{{\s*{currency}{number}\s*\}}",
        rf"Final answer:[ \t]*{currency}{number}",
    ]

    for pattern in patterns:
        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE,
        )
        if matches:
            return float(matches[-1].replace(",", ""))

    return None


def clean_solution(answer):
    return re.sub(r"<<[^>]*>>", "", answer).strip()


def prompt(question, four_shot=False):
    instruction = """Solve the math word problem step by step.
Keep each step brief and show the calculation.
End with #### followed by the final number only.
Do not include units or currency symbols after ####."""

    parts = [instruction]

    if four_shot:
        for i in range(min(4, len(demonstrations))):
            example = demonstrations[i]

            parts.append(
                f"""Question: {example['question'].strip()}
Solution:
{clean_solution(example['answer'])}"""
            )

    parts.append(
        f"""Question: {question.strip()}
Solution:
"""
    )

    return "\n\n".join(parts)


@torch.inference_mode()
def ask(content, allow_tools=False):
    messages = (
        [{"role": "user", "content": content}]
        if isinstance(content, str)
        else content
    )

    inputs = tokenizer.apply_chat_template(
        messages,
        tools=tools if allow_tools else None,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.1,
    )

    new_ids = output[0, inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_ids, skip_special_tokens=True)

    return response, len(new_ids)

In [7]:
print(prompt(test_questions[0]["question"], four_shot=True))

Solve the math word problem step by step.
Keep each step brief and show the calculation.
End with #### followed by the final number only.
Do not include units or currency symbols after ####.

Question: A deep-sea monster rises from the waters once every hundred years to feast on a ship and sate its hunger. Over three hundred years, it has consumed 847 people. Ships have been built larger over time, so each new ship has twice as many people as the last ship. How many people were on the ship the monster ate in the first hundred years?
Solution:
Let S be the number of people on the first hundred years’ ship.
The second hundred years’ ship had twice as many as the first, so it had 2S people.
The third hundred years’ ship had twice as many as the second, so it had 2 * 2S = 4S people.
All the ships had S + 2S + 4S = 7S = 847 people.
Thus, the ship that the monster ate in the first hundred years had S = 847 / 7 = 121 people on it.
#### 121

Question: Tobias is buying a new pair of shoes that 

In [9]:
from collections import Counter

def zero_shot(question):
    text, tokens = ask(prompt(question))
    return extract_answer(text), tokens, [text]

def four_shot(question):
    text, tokens = ask(prompt(question, four_shot=True))
    return extract_answer(text), tokens, [text]

def self_consistency(question):
    outputs = [ask(prompt(question, four_shot=True)) for _ in range(4)]
    answers = [extract_answer(text) for text, tokens in outputs]
    valid = [answer for answer in answers if answer is not None]

    chosen = Counter(valid).most_common(1)[0][0] if valid else None
    total_tokens = sum(tokens for text, tokens in outputs)
    texts = [text for text, tokens in outputs]

    return chosen, total_tokens, texts

def reflection(question):
    first, tokens1 = ask(prompt(question, four_shot=True))

    check = f"""Question: {question}

Previous solution:
{first}

Check the calculations and whether they answer the question.
Keep the solution if correct; otherwise fix it.
Write a short, complete solution.
End with #### followed by the final number only.
Do not include units or currency symbols after ####."""

    revised, tokens2 = ask(check)

    revised_answer = extract_answer(revised)
    chosen = (
        revised_answer
        if revised_answer is not None
        else extract_answer(first)
    )

    return chosen, tokens1 + tokens2, [first, revised]

In [10]:
from tqdm.auto import tqdm

methods = {
    "Zero-shot": zero_shot,
    "Four-shot": four_shot,
    "Self-consistency": self_consistency,
    "Reflection": reflection,
}

for name, method in methods.items():
    correct = 0

    for example in tqdm(development, desc=name):
        prediction, tokens, texts = method(example["question"])
        correct += prediction == extract_answer(example["answer"])

    print(f"{name}: {correct}/10")

Zero-shot:   0%|          | 0/10 [00:00<?, ?it/s]

Zero-shot: 1/10


Four-shot:   0%|          | 0/10 [00:00<?, ?it/s]

Four-shot: 4/10


Self-consistency:   0%|          | 0/10 [00:00<?, ?it/s]

Self-consistency: 4/10


Reflection:   0%|          | 0/10 [00:00<?, ?it/s]

Reflection: 4/10


In [11]:
reference_data = load_dataset("openai/gsm8k", "main", split="test[:30]")
references = [extract_answer(row["answer"]) for row in reference_data]


def show_result(name, rows, calls_per_question):
    n = len(references)
    correct = sum(rows[i][0] == references[i] for i in range(n))
    tokens = sum(row[1] for row in rows)
    calls = n * calls_per_question

    print(f"{name} | {correct}/{n} | {tokens} | {calls}")


print("Method | Accuracy | Generated tokens | Model calls")

zero_results = [
    zero_shot(row["question"])
    for row in tqdm(test_questions, desc="Zero-shot")
]
show_result("Zero-shot", zero_results, 1)

four_results = [
    four_shot(row["question"])
    for row in tqdm(test_questions, desc="Four-shot")
]
show_result("Four-shot", four_results, 1)

self_results = [
    self_consistency(row["question"])
    for row in tqdm(test_questions, desc="Self-consistency")
]
show_result("Self-consistency", self_results, 4)

reflection_results = [
    reflection(row["question"])
    for row in tqdm(test_questions, desc="Reflection")
]
show_result("Reflection", reflection_results, 2)

Method | Accuracy | Generated tokens | Model calls


Zero-shot:   0%|          | 0/30 [00:00<?, ?it/s]

Zero-shot | 5/30 | 7849 | 30


Four-shot:   0%|          | 0/30 [00:00<?, ?it/s]

Four-shot | 3/30 | 6433 | 30


Self-consistency:   0%|          | 0/30 [00:00<?, ?it/s]

Self-consistency | 5/30 | 23712 | 120


Reflection:   0%|          | 0/30 [00:00<?, ?it/s]

Reflection | 6/30 | 6683 | 60


In [13]:
for name, rows in [
    ("Self-consistency", self_results),
    ("Reflection", reflection_results),
]:
    for i in range(30):
        before = four_results[i][0] == references[i]
        after = rows[i][0] == references[i]

        if before != after:
            print("\nQuestion:", i + 1, "| Method:", name)
            print("Effect:", "helped" if after else "harmed")
            print("Reference:", references[i])
            print("Four-shot answer:", four_results[i][0])
            print("New answer:", rows[i][0])
            print("Question text:", test_questions[i]["question"])
            print("Four-shot solution:", four_results[i][2])
            print("New solutions:", rows[i][2])


Question: 14 | Method: Self-consistency
Effect: helped
Reference: 18.0
Four-shot answer: None
New answer: 18.0
Question text: Melanie is a door-to-door saleswoman. She sold a third of her vacuum cleaners at the green house, 2 more to the red house, and half of what was left at the orange house. If Melanie has 5 vacuum cleaners left, how many did she start with?
Four-shot solution: ["To find out how many vacuum cleaners Melanie started with, let's break down the information given:\n\n1. Melanie sold a third of her vacuum cleaners at the green house.\n   - Let \\( x \\) represent the total number of vacuum cleaners Melanie started with.\n   - At the green house, she sold \\( \\frac{x}{3} \\).\n\n2. She sold 2 more vacuum cleaners to the red house.\n   - This means she sold \\( \\frac{x}{3} + 2 \\) at the red house.\n\n3. She sold half of what was left at the orange house.\n   - The remaining vacuum cleaners before selling them were \\( x - (\\frac{x}{3} + 2) \\).\n   - Selling half of t

| Method | Accuracy | Generated tokens | Model calls |
|---|---:|---:|---:|
| Zero-shot | 5/30 (16.7%) | 7,849 | 30 |
| Four-shot | 3/30 (10.0%) | 6,433 | 30 |
| Self-consistency | 5/30 (16.7%) | 23,712 | 120 |
| Reflection | 6/30 (20.0%) | 6,683 | 60 |

Self-consistency and reflection used the four-shot prompt for their initial solutions. Reflection achieved the highest accuracy, while self-consistency matched zero-shot accuracy at a much higher cost.

**Case 1 — Reflection helped (Question 1):** The initial solution incorrectly calculated Janet’s earnings as $32. The revision correctly calculated the remaining eggs as 16 − 3 − 4 = 9 and the earnings as 9 × 2 = $18. Here, the additional call corrected the calculation.

**Case 2 — Reflection performed worse than four-shot (Question 10):** Four-shot correctly calculated Eliza’s earnings as 40 × 10 + 5 × 12 = $460. Reflection’s initial solution instead added only $12 in overtime pay, producing $412, and the revision failed to correct it. This shows that extra inference does not guarantee improvement. The revision did not introduce this error; it left an existing error unresolved.

**Scoring limitation:** Answers without a recognized final-answer format were recorded as `None`, even when their reasoning was correct. Therefore, the reported accuracy reflects both mathematical correctness and answer formatting.

**Best accuracy–cost tradeoff:** Reflection provides the best balance in this run: it achieves the highest accuracy with fewer generated tokens than zero-shot, although it requires twice as many model calls. Zero-shot is preferable when minimizing calls is the priority. Self-consistency’s additional cost produced no accuracy gain over zero-shot.

# Question 2

In [14]:
%pip -q install langchain-core

In [15]:
import ast
import operator
from tqdm.auto import tqdm
import json
import re
from langchain_core.tools import tool

The calculator evaluates basic arithmetic expressions with `ast`, without using unrestricted `eval()` or `exec()`.

In [16]:
ops = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.USub: operator.neg,
}

@tool
def calculator(expression: str) -> str:
    """Calculate an arithmetic expression using +, -, *, /."""
    def calculate(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value
        if isinstance(node, ast.BinOp):
            return ops[type(node.op)](calculate(node.left), calculate(node.right))
        if isinstance(node, ast.UnaryOp):
            return ops[type(node.op)](calculate(node.operand))
        raise ValueError("Invalid expression")

    try:
        return str(calculate(ast.parse(expression, mode="eval").body))
    except Exception as e:
        return f"Error: {e}"

In [17]:
print(calculator.invoke({"expression": "12 + 8 * 3"}))

36


The tool schema tells Qwen how to request calculations. The prompt limits calculator use to three calls.

In [23]:
tools = [{
    "type": "function",
    "function": {
        "name": calculator.name,
        "description": calculator.description,
        "parameters": calculator.args_schema.model_json_schema(),
    }
}]


agent_prompt = """
Solve the math word problem in brief steps.
Choose the quantities needed to answer the question.
Use the calculator for arithmetic, at most 3 times and one call at a time.
Do not include words or units in calculator expressions.
Use the calculator results and show the calculations.
End with #### followed by the final number only, without units or currency symbols.
"""

The agent sends calculator results back to Qwen and records generated tokens, model calls, and tool calls.

In [26]:
def agent(question):
    messages = [
        {"role": "system", "content": agent_prompt},
        {"role": "user", "content": question},
    ]

    tokens = 0
    model_calls = 0
    tool_calls = 0
    trace = []

    while True:
        allow_tools = tool_calls < 3

        if not allow_tools:
            messages.append({
                "role": "user",
                "content": (
                    "No more tool calls. Use the available results. "
                    "End with #### followed by the final number only."
                ),
            })

        response, n = ask(messages, allow_tools=allow_tools)
        tokens += n
        model_calls += 1
        trace.append(("Model", response))

        match = re.search(
            r"<tool_call>\s*(.*?)\s*</tool_call>",
            response,
            flags=re.S,
        )

        if not match or not allow_tools:
            break

        try:
            call = json.loads(match.group(1))

            if not isinstance(call, dict):
                break

            if call.get("name") != calculator.name:
                break

            arguments = call["arguments"]
            if isinstance(arguments, str):
                arguments = json.loads(arguments)

            if not isinstance(arguments, dict):
                break

            expression = arguments["expression"]
            if not isinstance(expression, str):
                break

        except (ValueError, KeyError, TypeError):
            break

        tool_calls += 1

        try:
            result = calculator.invoke({"expression": expression})
        except Exception as exc:
            result = f"Calculator error: {exc}"

        trace.append(("Calculator", result))

        messages.append({
            "role": "assistant",
            "content": response[:match.start()].strip(),
            "tool_calls": [{
                "type": "function",
                "function": {
                    "name": calculator.name,
                    "arguments": {"expression": expression},
                },
            }],
        })

        messages.append({
            "role": "tool",
            "name": calculator.name,
            "content": str(result),
        })

    answer = extract_answer(response)

    if answer is None:
        messages.append({"role": "assistant", "content": response})
        messages.append({
            "role": "user",
            "content": (
                "Repeat your final answer on one line as #### <number>. "
                "Do not solve again or call tools. "
                "Do not include units or currency symbols."
            ),
        })

        response, n = ask(messages, allow_tools=False)
        tokens += n
        model_calls += 1
        trace.append(("Final", response))

        answer = extract_answer(response)

    return answer, tokens, model_calls, tool_calls, trace

In [27]:
dev_results = [
    agent(row["question"])
    for row in tqdm(development, desc="Agent Development")
]

correct = sum(
    dev_results[i][0] == extract_answer(development[i]["answer"])
    for i in range(10)
)

print(f"Calculator Agent: {correct}/10")

Agent Development:   0%|          | 0/10 [00:00<?, ?it/s]

Calculator Agent: 4/10


In [33]:
for i, r in enumerate(dev_results):
    print(i + 1, "Prediction:", r[0], "Reference:", extract_answer(development[i]["answer"]))

for i in range(10):
    print("\nQuestion:", i + 1)
    print("Prediction:", dev_results[i][0])
    print("Tool calls:", dev_results[i][3])

    for step, text in dev_results[i][4]:
        print(step, ":", text)

    print("-" * 40)

1 Prediction: 72.0 Reference: 72.0
2 Prediction: 10.0 Reference: 10.0
3 Prediction: 77.5 Reference: 5.0
4 Prediction: 68.0 Reference: 42.0
5 Prediction: 4408.0 Reference: 624.0
6 Prediction: 38.61 Reference: 35.0
7 Prediction: 48.0 Reference: 48.0
8 Prediction: 16.0 Reference: 16.0
9 Prediction: 137.0 Reference: 41.0
10 Prediction: 360.0 Reference: 990.0

Question: 1
Prediction: 72.0
Tool calls: 0
Model : To solve this problem, we need to calculate the total number of clips sold in April and May separately and then add them together.

1. Calculate the number of clips sold in April: \( \text{Clips sold in April} = 48 \)
2. Calculate the number of clips sold in May: \( \text{Clips sold in May} = \frac{\text{Clips sold in April}}{2} \)

Let's perform these calculations:

1. Clips sold in April: \( 48 \)
2. Clips sold in May: \( \frac{48}{2} = 24 \)

Now, let's sum up the clips sold in both months:

\[ \text{Total clips sold} = \text{Clips sold in April} + \text{Clips sold in May} \]
\[ \t

In [32]:
agent_results = [
    agent(row["question"])
    for row in tqdm(test_questions, desc="Calculator Agent")
]

results = [
    ("Zero-shot", zero_results, 1),
    ("Four-shot", four_results, 1),
    ("Self-consistency", self_results, 4),
    ("Reflection", reflection_results, 2),
]

print("Method | Accuracy | Generated tokens | Model calls | Tool calls")

for name, rows, calls_per_question in results:
    correct = sum(rows[i][0] == references[i] for i in range(30))
    tokens = sum(r[1] for r in rows)

    print(f"{name} | {correct}/30 | {tokens} | {30 * calls_per_question} | 0")

correct = sum(agent_results[i][0] == references[i] for i in range(30))
tokens = sum(r[1] for r in agent_results)
model_calls = sum(r[2] for r in agent_results)
tool_calls = sum(r[3] for r in agent_results)

print(f"Calculator Agent | {correct}/30 | {tokens} | {model_calls} | {tool_calls}")

Calculator Agent:   0%|          | 0/30 [00:00<?, ?it/s]

Method | Accuracy | Generated tokens | Model calls | Tool calls
Zero-shot | 5/30 | 7849 | 30 | 0
Four-shot | 3/30 | 6433 | 30 | 0
Self-consistency | 5/30 | 23712 | 120 | 0
Reflection | 6/30 | 6683 | 60 | 0
Calculator Agent | 11/30 | 7183 | 67 | 9


The agent uses a restricted AST-based calculator for arithmetic, without unrestricted `eval()` or `exec()`. It receives calculation results or errors and is limited to three calculator calls per question.

| Method | Correct / 30 | Accuracy | Generated tokens | Model calls | Calculator calls |
|---|---:|---:|---:|---:|---:|
| Zero-shot | 5 | 16.7% | 7,849 | 30 | 0 |
| Four-shot | 3 | 10.0% | 6,433 | 30 | 0 |
| Self-consistency | 5 | 16.7% | 23,712 | 120 | 0 |
| Reflection — best Q1 method | 6 | 20.0% | 6,683 | 60 | 0 |
| **Calculator Agent** | **11** | **36.7%** | **7,183** | **67** | **9** |

**Comparison:** The agent answered five more questions correctly than reflection, increasing accuracy by 16.7 percentage points. It used approximately 7.5% more generated tokens and seven additional model calls. This gives the agent a better accuracy–cost tradeoff in this run, although the comparison does not isolate the calculator’s contribution.

**Failure 1 — Wrong expression (development Question 3):** The agent submitted `100 - ((15 + 2*15) / 2)`, obtaining 77.5. The correct expression was `100/2 - 15 - 2*15 = 5`. The calculator worked correctly; the model halved the wrong quantity.

**Failure 2 — Wrong formulation and arithmetic (development Question 5):** The model treated four letters per week as four per day, incorrectly adding a factor of seven. It then calculated `84 × 52` as 4408 rather than 4368. No calculator was used. A calculator could fix the multiplication, but the correct formulation was still `3 × 2 × 2 × 52 = 624`.

**Why tool success is insufficient:** A calculator evaluates the expression it receives. It cannot verify that the expression matches the word problem, and the model may also misinterpret or ignore its result.

# Question 3

One potential methodological flaw in Ferber et al.’s study is that the patient simulator treats undocumented symptoms as absent. In the Supplementary Information, Section 17 (“Patient agent instructions”), page 30, the authors instruct the simulator:

> “As another example, if questioned about a symptom not included in the summary, state that you do not have that symptom.”

This instruction conflates missing information with a confirmed negative finding. In clinical documentation, an unmentioned symptom may not have been assessed or recorded. For example, if breathlessness is not mentioned in a patient’s history, the simulator would deny it when questioned, although the record does not establish its absence.

This could distort diagnostic reasoning by providing unsupported negative evidence. It might make some cases artificially easier by allowing alternative diagnoses to be excluded, while misleading decision-making in others. Consequently, performance in this simulation may not generalize to real encounters involving incomplete information. Since both MIRA and the physicians interacted with the same simulator, this limitation does not, by itself, establish an unfair advantage for MIRA.

I suggest distinguishing three information states: present, explicitly absent, and unknown or undocumented. The simulator should deny a symptom only when its absence is documented. A paired sensitivity analysis should then repeat the same cases under the original and revised instructions for both MIRA and physicians. Comparing diagnostic accuracy and clinical decisions would reveal how strongly the conclusions depend on this assumption.
